# Measurement Example

This notebook shows basic usage of `qubex.measurement.Measurement`.

- Create a Measurement instance
- Load/connect devices
- Prepare a pulse sequence
- Run `measure` / `execute`

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from qubex.measurement import Measurement
from qubex.pulse import Blank, FlatTop, Gaussian, PulseSchedule

In [ ]:
# Build an instance without touching hardware (safe for offline docs).
meas = Measurement(
    chip_id="64Qv3",
    qubits=["Q62", "Q63"],
    config_dir="/home/shared/qubex-config/64Qv3/config",
    params_dir="/home/shared/qubex-config/64Qv3/params",
)

In [ ]:
# Connect to devices (requires hardware access).
meas.connect()

In [ ]:
# Measure noise on all qubits for 10,000 ns.
result = meas.measure_noise(
    meas.qubits,
    duration=10000,
)
result.plot()
result.data

In [ ]:
# Basic measurement from waveform mapping (2 ns sampling period).
waveforms = {
    "Q62": np.full(10, 0.03),
    "Q63": np.full(10, 0.03),
}

# `measure` will automatically add readout pulses at the end of the sequence.

# Single-shot measurement
result = meas.measure(
    waveforms,
    mode="single",
    shots=1024,
    plot=True,
)
result.plot()

# Average measurement
result = meas.measure(
    waveforms,
    mode="avg",
    shots=1024,
)
result.plot()

In [ ]:
# Measurement from PulseSchedule
with PulseSchedule() as seq1:
    seq1.add("Q62", Gaussian(duration=32, amplitude=0.03, sigma=8))
    seq1.add("Q63", Gaussian(duration=32, amplitude=0.03, sigma=8))
    seq1.add("Q62", Blank(duration=16))
    seq1.barrier()
    seq1.add("RQ62", FlatTop(duration=256, amplitude=0.1, tau=32))
    seq1.add("RQ63", FlatTop(duration=256, amplitude=0.1, tau=32))
seq1.plot()

result = meas.execute(
    seq1,
    mode="single",
    shots=1024,
)
result.plot()

In [ ]:
# Basic measurement from waveform mapping (2 ns sampling period).
with PulseSchedule() as seq2:
    for _ in range(3):
        seq2.call(seq1)
seq2.plot()

result = meas.execute(
    seq2,
    mode="single",
    shots=1024,
    plot=True,
    add_pump_pulses=True,
)
result.plot()